# HanziGen - 字型生成训练（本地机版）

> 本 notebook 适用于**在自己的电脑上**运行训练，无需任何云平台。
>
> 与云端版（`hanzigen_cloudstudio.ipynb` / `hanzigen_colab.ipynb` / `hanzigen_moda.ipynb`）的区别：
> - **硬件档位用 conservative（本地稳妥档）**：workers 留 2 核余量、预取保守、显存预留更稳，避免影响你日常使用、防止 OOM。
> - **内置硬件适配性检测**：Cell 1 会自动判断你的电脑是否满足训练要求（需要 NVIDIA GPU 或 Apple Silicon；AMD 核显/纯 CPU 会提示无法训练）。
> - 无需挂载网盘、无需 clone，**直接在当前项目目录运行**。

## 前置要求

1. 已安装 Python 3.10+；依赖**会在 Cell 1 自动补装**（也可手动 `pip install -r requirements.txt`）。在 AutoDL / 云服务器等新环境上直接跑即可，若报错缺包请重跑 Cell 1。
2. 有可用的训练加速器：**NVIDIA GPU（建议显存 >= 8GB）** 或 **Apple Silicon (MPS)**
   - 仅 AMD 核显 / 纯 CPU 的机器**无法运行训练**（本项目模型依赖 CUDA/MPS）
3. 目标字体 `.ttf` / `.otf` 已放入 `fonts/` 目录

## 流程

```
Cell 0: 配置字体名 + 阶段开关 + 补字基准 + 显存冗余量
Cell 1: 环境自检 + 硬件适配性检测 + 断连自检（可重复运行）
Cell 2: 数据准备（分析字体 → 生成数据集 → 提取字集）
Cell 3: 训练 VQ-VAE（conservative 档）
Cell 4-前置操作: 离线放置 VGG16 权重（可选，联网下载慢时运行）
Cell 4: 训练 LDM（conservative 档）
Cell 5: 推理生成 + 评估指标（GPU）
Cell 6: SVG 转换（纯 CPU，可切无卡模式）
```

---
## Cell 0: 配置参数

> **只改这里！** 填你放入 `fonts/` 的字体文件名，并选择要补哪些字（补字基准）。

In [ ]:
# OpenMP 线程数：部分云镜像预置的 OMP_NUM_THREADS 值非法，会导致 libgomp 警告并按默认开满全部 CPU 核
# （须在 import torch/numpy 之前设置；注意用直接赋值强制覆盖——镜像里预置的值往往本身就是非法的，
#   setdefault 会因"键已存在"而跳过，导致非法值传给训练子进程）
# 取值自动适配：实际可用核数的一半、封顶 8（大机器足够用，小机器避免与 DataLoader workers 抢核）
import os
try:
    _cpu = len(os.sched_getaffinity(0))          # Linux：本进程实际可用的核数
except AttributeError:
    _cpu = os.cpu_count() or 4                    # Windows / macOS 回退
os.environ["OMP_NUM_THREADS"] = str(min(8, max(1, _cpu // 2)))

# ==================== 修改你的字体文件名 ====================
TARGET_FONT = "myfont.ttf"    # 改成你放入 fonts/ 的字体名
# ==========================================================

FONT_NAME = TARGET_FONT.rsplit(".", 1)[0]

# ==================== 训练阶段开关（默认全开）====================
DO_DATA_PREP = True       # Cell 2: 数据准备
DO_TRAIN_VQVAE = True     # Cell 3: 训练 VQ-VAE
DO_TRAIN_LDM = True       # Cell 4: 训练 LDM
DO_INFERENCE = True       # Cell 5: 推理+指标+转SVG
# ==========================================================

# ==================== 补字基准（Cell 5 推理阶段生效）====================
# "jf7000" = jf7000 当务字集缺失字（默认，约 8,349 字基准，项目原生设计）
# "unihan" = Unihan 全字集缺失字（9 万+ 字基准，生成量大，注意耗时）
# "gbk"    = GBK 简体标准字符集缺失字（20,902 字基准，简体用户推荐）
# "gb2312" = 仅 GB2312 简体核心字（6,763 字基准，范围最保守）
CHARSET_BASE = "jf7000"
# ==========================================================

# ==================== 参考字体设置（数据准备与推理共用）====================
REFERENCE_FONTS_DIR = "fonts/jigmo"   # 参考字体目录（提供字形结构供模型学习/生成）
REF_FONT_PRIORITY = ""                # 覆盖优先级：逗号分隔文件名，从高到低
                                      #   空 = 按文件名顺序（jigmo → jigmo2 → jigmo3）
                                      #   同一字符被多个字体覆盖时，排在前面的优先生效
                                      #   例：只让 jigmo.ttf 提供常用简繁字形 → "jigmo.ttf"
REF_FONT_MODE = "first"               # first=优先字体先命中（推荐）
                                      # last=旧行为（后写入的字体覆盖先写入的）
REF_FONT_STRICT = False               # 严格模式：True=只使用优先级列表中的字体（风格彻底统一）
                                      #   注意：若新参考未 100% 覆盖缺字表，参考有缺口会导致
                                      #   推理报错（数据集要求 gt/ref 文件名完全一致），
                                      #   因此一般保持 False 让 jigmo 兜底
# ==========================================================

# ==================== 显存冗余量（Cell 3 训练 VQ-VAE 生效）====================
# 显存预留比例 = 允许用于训练的那部分显存占比；留的冗余 = 1 - 该比例
#   None  = 跟随档位默认（本地 conservative 档 0.85，即留 15% 冗余）
#   0.70  = 留 30% 冗余（显存较小 / 常 OOM / 训练时还要用这台电脑，推荐先试这个）
#   0.92  = 只留 8% 冗余（显存充裕且独占，追求速度）
# 取值范围 0.1-1.0。仅影响 VQ-VAE 的 batch 推算；LDM 在 latent 空间训练不受影响。
VRAM_RESERVE_FRACTION = None
# ==========================================================

STATE_FILE = "colab_state.json"

print(f"目标字体: {TARGET_FONT}")
print(f"字体名称: {FONT_NAME}")
print(f"阶段开关: 数据准备={DO_DATA_PREP} VQVAE={DO_TRAIN_VQVAE} LDM={DO_TRAIN_LDM} 推理={DO_INFERENCE}")
print(f"补字基准: {CHARSET_BASE}")
print(f"显存预留比例: {VRAM_RESERVE_FRACTION if VRAM_RESERVE_FRACTION is not None else 'auto（跟随 conservative 档 0.85）'}")

---
## Cell 1: 依赖自检 + 硬件适配性检测 + 断连自检

> 依次完成：依赖自检补装 → **字体检查与所有 `scripts/*.sh` 字体路径改写** → 硬件适配性检测 → 断连自检与续训改写。可重复运行（幂等）。
>
> ⚠️ **Cell 0 改了 `TARGET_FONT` 后必须重跑本 Cell**，否则 Cell 2 调用的脚本里仍是旧的字体名。

In [ ]:
# ===== 0. 依赖自检与自动补装（幂等：只装缺失的）=====
import importlib, subprocess, sys, shutil, os, json, re, glob

# import 名 → pip 包名（两者不一致的需显式映射）
REQUIRED_PACKAGES = {
    "torch": "torch",            # 不自动装：可能覆盖平台预装的 CUDA 版，缺失时只提示
    "torchvision": "torchvision",
    "rich": "rich",
    "fontTools": "fonttools",
    "lpips": "lpips",
    "matplotlib": "matplotlib",
    "skimage": "scikit-image",
    "cleanfid": "clean-fid",
    "potrace": "potracer",       # 注意：pip 包名 potracer，import 名是 potrace
    "tqdm": "tqdm",
    "einops": "einops",
    "PIL": "pillow",
    "numpy": "numpy",
}
NO_AUTO_INSTALL = {"torch", "torchvision"}   # 装错版本代价大，只提示

def _can_import(mod: str) -> bool:
    try:
        importlib.import_module(mod)
        return True
    except Exception:
        return False

print("===== 依赖自检 =====")
print(f"  notebook kernel: {sys.executable}")
AUTO_INSTALL_DEPS = True   # 设为 False 可跳过自动安装（已用 conda 精心配好环境、不想被 pip 改动时）
if AUTO_INSTALL_DEPS and os.path.exists("requirements.txt"):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    print(f"  requirements.txt: {'安装完成' if r.returncode == 0 else '未完全成功，继续按 import 自检补装'}")

missing = [(m, p) for m, p in REQUIRED_PACKAGES.items() if not _can_import(m)]
if not missing:
    print("  依赖已齐全")
else:
    print(f"  缺失模块: {[m for m, _ in missing]}")
    to_install = [p for m, p in missing if m not in NO_AUTO_INSTALL]
    if AUTO_INSTALL_DEPS and to_install:
        print(f"  正在安装: {to_install}")
        r = subprocess.run([sys.executable, "-m", "pip", "install", *to_install],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print("  [ERROR] pip 安装失败，输出尾部：")
            print("\n".join((r.stdout + r.stderr).strip().splitlines()[-15:]))
    still = [m for m in REQUIRED_PACKAGES if not _can_import(m)]
    if still:
        print(f"  [WARN] 仍未就绪: {still}")
        if "torch" in still:
            print("        torch 未安装，请按平台要求手动安装对应 CUDA 版本，例如：")
            print("        pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124")
        if "potrace" in still:
            print("        potracer 缺失会导致 Cell 5 的 SVG 转换失败（训练不受影响）。")
            print("        它是 C 扩展包，编译失败时先装系统库：")
            print("          apt-get update && apt-get install -y build-essential libpotrace-dev")
            print("        若 apt 源里没有 libpotrace-dev，可试 conda：conda install -c conda-forge potrace")

# 关键：校验 shell 里的 python 与 notebook kernel 是否同一个解释器（autodl / conda 常见坑）
sh_python = shutil.which("python") or shutil.which("python3")
if sh_python and os.path.abspath(sh_python) != os.path.abspath(sys.executable):
    print(f"  [WARN] 终端 python 与 notebook kernel 不一致：\n        {sh_python}\n        {sys.executable}")
    print("        scripts/*.sh 内部用终端 python 执行，包装到 kernel 环境也会仍然报找不到模块。")
    print("        解决：conda activate 对应环境后再启动 jupyter；或把 kernel 的 python 软链到 which python 的位置。")
else:
    print(f"  python 解释器一致: {sh_python}")

# ===== 1. 检查目标字体（找不到就在常见目录搜，找到后复制到 fonts/）=====
print("===== 检查字体 =====")
font_path = f"fonts/{TARGET_FONT}"

def find_font_file(filename: str, roots, max_depth=2):
    """在 roots 下按有限深度搜索字体文件（不遍历整个数据盘）"""
    skip = {"node_modules", "__pycache__", "venv", ".venv", "env", "AppData", "Library"}
    for root in roots:
        if not os.path.isdir(root):
            continue
        base_depth = root.rstrip(os.sep).count(os.sep)
        for dirpath, dirnames, filenames in os.walk(root):
            if dirpath.rstrip(os.sep).count(os.sep) - base_depth >= max_depth:
                dirnames[:] = []
            else:
                dirnames[:] = [d for d in dirnames if d not in skip and not d.startswith(".")]
            for f in filenames:
                if f.lower() == filename.lower():
                    return os.path.join(dirpath, f)
    return None

print(f"  fonts/ 目录内容: {sorted(os.listdir('fonts')) if os.path.isdir('fonts') else '(目录不存在)'}")

if not os.path.exists(font_path):
    home = os.path.expanduser("~")
    searched = [os.getcwd(), os.path.dirname(os.getcwd()), home,
                os.path.join(home, "Downloads"), os.path.join(home, "Desktop")]
    found = find_font_file(TARGET_FONT, searched)
    if found:
        print(f"  [OK] 找到字体: {found} → 复制到 fonts/")
        os.makedirs("fonts", exist_ok=True)
        shutil.copy2(found, font_path)
    else:
        raise FileNotFoundError(
            f"\n字体文件 '{TARGET_FONT}' 在 fonts/ 与常见目录中都找不到！\n\n请检查:\n"
            f"  1. Cell 0 的 TARGET_FONT 是否填成了你的字体文件名（当前: '{TARGET_FONT}'）\n"
            f"     —— 占位默认值 myfont.ttf 必须改掉\n"
            f"  2. 字体是否已上传到 fonts/ 目录（当前项目根: {os.getcwd()}）\n"
            f"  3. 文件名大小写是否完全一致"
        )
else:
    print(f"  [OK] 目标字体已就绪: {font_path}")

# ===== 2. 把字体路径写进所有 .sh 脚本（否则脚本里仍是默认的 myfont.ttf）=====
changed = []
for sh_file in sorted(glob.glob("scripts/*.sh")):
    with open(sh_file, "r", encoding="utf-8") as f:
        content = f.read()
    # 允许中文 / 空格文件名；不跨引号，避免误伤 fonts/jigmo/ 这类目录引用
    new_content = re.sub(r'fonts/[^\x22\x27]+?\.(?:ttf|otf|TTF|OTF)', f'fonts/{TARGET_FONT}', content)
    if new_content != content:
        with open(sh_file, "w", encoding="utf-8") as f:
            f.write(new_content)
        changed.append(sh_file)
print(f"  脚本字体路径已统一为 fonts/{TARGET_FONT}（改写 {len(changed)} 个脚本）")

# ===== 2.5 参考字体：目录/优先级写入脚本与环境变量（训练与推理子进程继承）=====
os.environ["HANZIGEN_REF_FONT_PRIORITY"] = REF_FONT_PRIORITY
os.environ["HANZIGEN_REF_FONT_MODE"] = REF_FONT_MODE
os.environ["HANZIGEN_REF_FONT_STRICT"] = "1" if REF_FONT_STRICT else "0"
ref_changed_scripts = 0
for sh_file in sorted(glob.glob("scripts/*.sh")):
    with open(sh_file, "r", encoding="utf-8") as f:
        content = f.read()
    new_content = re.sub(r'REFERENCE_FONTS_DIR="[^"]*"',
                         f'REFERENCE_FONTS_DIR="{REFERENCE_FONTS_DIR}"', content)
    if new_content != content:
        with open(sh_file, "w", encoding="utf-8") as f:
            f.write(new_content)
        ref_changed_scripts += 1
print(f"  参考字体: {REFERENCE_FONTS_DIR} | 优先级: {REF_FONT_PRIORITY or '默认文件名顺序'} | 模式: {REF_FONT_MODE}（改写 {ref_changed_scripts} 个脚本）")

# ===== 3. 准备 Jigmo 参考字体（训练必需：缺失时 data/reference 不会生成）=====
print("===== 准备 Jigmo 参考字体 =====")
JIGMO_FILES = ["jigmo.ttf", "jigmo2.ttf", "jigmo3.ttf"]
JIGMO_DIR = "fonts/jigmo"
JIGMO_ZIP_URL = "https://kamichikoichi.github.io/jigmo/Jigmo-20250912.zip"

def find_first_match(roots, predicate, max_depth=2):
    """在 roots 下按有限深度找第一个满足 predicate(文件名) 的文件（不遍历整个数据盘）"""
    skip = {"node_modules", "__pycache__", "venv", ".venv", "env", "AppData", "Library"}
    for root in roots:
        if not os.path.isdir(root):
            continue
        base_depth = root.rstrip(os.sep).count(os.sep)
        for dirpath, dirnames, filenames in os.walk(root):
            if dirpath.rstrip(os.sep).count(os.sep) - base_depth >= max_depth:
                dirnames[:] = []
            else:
                dirnames[:] = [d for d in dirnames if d not in skip and not d.startswith(".")]
            for f in filenames:
                if predicate(f):
                    return os.path.join(dirpath, f)
    return None

def validate_font_file(fpath):
    try:
        from fontTools.ttLib import TTFont
        f = TTFont(fpath)
        if "cmap" not in f:
            return False, "缺少 cmap 表"
        return True, f"OK ({len(f.getBestCmap())} glyphs)"
    except Exception as e:
        return False, str(e)[:80]

def extract_jigmo(zip_path):
    """从 Jigmo ZIP 中取出 jigmo.ttf / jigmo2.ttf / jigmo3.ttf（忽略包内目录结构）"""
    import zipfile
    got = []
    with zipfile.ZipFile(zip_path) as zf:
        for name in zf.namelist():
            base = os.path.basename(name).lower()
            if base in JIGMO_FILES:
                with zf.open(name) as src, open(os.path.join(JIGMO_DIR, base), "wb") as dst:
                    shutil.copyfileobj(src, dst)
                got.append(base)
    return got

def download_jigmo():
    import urllib.request
    print(f"  正在下载: {JIGMO_ZIP_URL}")
    try:
        with urllib.request.urlopen(JIGMO_ZIP_URL, timeout=120) as resp:
            data = resp.read()
        if len(data) < 10000:
            raise ValueError(f"下载数据异常 ({len(data)} bytes)")
        tmp_zip = os.path.join(JIGMO_DIR, "_jigmo_tmp.zip")
        with open(tmp_zip, "wb") as f:
            f.write(data)
        got = extract_jigmo(tmp_zip)
        os.remove(tmp_zip)
        print(f"  已从官方 ZIP 提取: {got}")
        return bool(got)
    except Exception as e:
        print(f"  [ERROR] 下载失败: {e}")
        return False

os.makedirs(JIGMO_DIR, exist_ok=True)
_home = os.path.expanduser("~")
_roots = [os.getcwd(), os.path.dirname(os.getcwd()), _home,
          os.path.join(_home, "Downloads"), os.path.join(_home, "Desktop")]

for fname in JIGMO_FILES:
    fpath = os.path.join(JIGMO_DIR, fname)
    if os.path.exists(fpath):
        ok, msg = validate_font_file(fpath)
        if ok:
            continue
        print(f"  [WARN] {fname} 无效（{msg}），将重新获取")
        os.remove(fpath)
    # ① 本地已有解压好的 jigmo*.ttf → 直接复制
    src = find_first_match(_roots, lambda f, _n=fname: f.lower() == _n)
    if src and os.path.abspath(src) != os.path.abspath(fpath):
        shutil.copy2(src, fpath)
        print(f"  [OK] 复制 {fname} ← {src}")
        continue
    # ② 本地有 Jigmo*.zip → 解压
    zpath = find_first_match(_roots, lambda f: f.lower().endswith(".zip") and "jigmo" in f.lower())
    if zpath and fname in extract_jigmo(zpath):
        print(f"  [OK] 从 {zpath} 解压出 {fname}")
        continue
    # ③ 联网下载官方 ZIP
    if not download_jigmo():
        break

missing_jigmo = []
for fname in JIGMO_FILES:
    fpath = os.path.join(JIGMO_DIR, fname)
    if not os.path.exists(fpath):
        missing_jigmo.append(fname)
        continue
    ok, msg = validate_font_file(fpath)
    if not ok:
        print(f"  [ERROR] {fname} 校验失败（{msg}）")
        os.remove(fpath)
        missing_jigmo.append(fname)
if missing_jigmo:
    raise RuntimeError(
        f"\nJigmo 参考字体缺失: {missing_jigmo}\n"
        f"  没有参考字体就无法生成 data/reference，训练无法进行。\n"
        f"  手动处理：从 https://kamichikoichi.github.io/jigmo/ 下载 Jigmo 压缩包，\n"
        f"  把其中的 jigmo.ttf / jigmo2.ttf / jigmo3.ttf 放到 {os.path.abspath(JIGMO_DIR)}/ 后重跑本 Cell。"
    )
for fname in JIGMO_FILES:
    ok, msg = validate_font_file(os.path.join(JIGMO_DIR, fname))
    print(f"  [{'OK' if ok else 'ERROR'}] {fname}: {msg}")

import torch
from utils.hardware import check_training_viability, detect_hardware

print("===== 硬件适配性检测 =====")
info = detect_hardware()
print(f"  CPU 核数: {info['cpu_cores']}")
if info["gpu_available"]:
    print(f"  GPU: {info['gpu_name']} ({info['vram_gb']:.1f} GB)")
elif info["mps_available"]:
    print(f"  GPU: {info['gpu_name']}（Apple Silicon MPS）")
else:
    print("  GPU: 未检测到（仅 CPU / 核显）")
print(f"  PyTorch: {torch.__version__} | CUDA: {torch.version.cuda or '无'}")

viability = check_training_viability()
if viability["viable"]:
    print(f"\n[OK] 配置可训练：{viability['reason']}")
else:
    print(f"\n[无法训练] {viability['reason']}")
    print("  → 请改用带 NVIDIA GPU 或 Apple Silicon 的机器，或使用云端 GPU 实例")
    print("    （云端请用 hanzigen_cloudstudio.ipynb / hanzigen_colab.ipynb / hanzigen_moda.ipynb）")
    print("  → 数据准备（Cell 2）与 SVG 转换（Cell 5 后半）仍可在此机器运行，但训练（Cell 3/4）无法执行。")

# ===== 断连自检 + 字体切换检测 + 自动续训改写 =====
print("\n===== 断连自检 + 字体切换检测 =====")
os.makedirs("checkpoints", exist_ok=True)

# ===== 参考字体变更检测：参考字形变了 → 训练数据与模型都建立在旧参考上 =====
def _ref_fingerprint() -> list:
    d = REFERENCE_FONTS_DIR
    if not os.path.isdir(d):
        return []
    return sorted((f, os.path.getsize(os.path.join(d, f)))
                  for f in os.listdir(d) if f.lower().endswith((".ttf", ".otf")))

_ref_sig = _ref_fingerprint()
_prev_ref_sig = state.get("ref_fonts_sig")
if _prev_ref_sig is not None and _prev_ref_sig != _ref_sig:
    print("  [参考字体变更] 检测到参考字体与上次训练时不一致！")
    print("    · 训练数据(data/)与模型都是在旧参考字形上产生的，参考变了就不再匹配")
    print("    · 已自动作废数据准备标记（Cell 2 将清空 data/ 重建数据集）")
    print("    · 强烈建议删除检查点从零训练（续训会把新旧参考的知识混在一起）：")
    print(f"        rm -f checkpoints/vqvae_{FONT_NAME}.pth checkpoints/ldm_{FONT_NAME}.pth")
    print("        rm -f checkpoints/vqvae_*.pth_last checkpoints/ldm_*.pth_last")
    state["data_prep_done"] = False
elif _prev_ref_sig is None:
    print("  参考字体指纹已记录（首次运行）")
state["ref_fonts_sig"] = _ref_sig


def load_state() -> dict:
    if os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            return {}
    return {}

def save_state(state: dict) -> None:
    with open(STATE_FILE, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)

state = load_state()

# 字体切换检测：数据集状态与字体名绑定，防止新旧字体数据混合
prev_data_font = state.get("data_font")
if prev_data_font and prev_data_font != FONT_NAME:
    print(f"  [字体切换] {prev_data_font} → {FONT_NAME}")
    print("    · Cell 2 将重新执行数据准备（旧字体图像会被自动删除）")
    state["data_prep_done"] = False
    state["data_font"] = None

state.setdefault("font", FONT_NAME)
save_state(state)

vqvae_ckpt = f"checkpoints/vqvae_{FONT_NAME}.pth"
ldm_ckpt = f"checkpoints/ldm_{FONT_NAME}.pth"
have_vqvae = os.path.exists(vqvae_ckpt)
have_ldm = os.path.exists(ldm_ckpt)

print(f"  数据准备: {'已完成' if state.get('data_prep_done') else '未完成'}（绑定字体: {state.get('data_font') or '无'}）")
print(f"  VQ-VAE 检查点: {'存在' if have_vqvae else '不存在'}")
print(f"  LDM 检查点:    {'存在' if have_ldm else '不存在'}")

# 自动精确续训：改写 local 脚本的 RESUME_FROM
def _set_resume_from(sh_path: str, ckpt: str) -> None:
    with open(sh_path, "r", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r'RESUME_FROM="[^"]*"', f'RESUME_FROM="{ckpt}"', content)
    with open(sh_path, "w", encoding="utf-8") as f:
        f.write(content)

_set_resume_from("scripts/train_vqvae_local.sh", vqvae_ckpt if have_vqvae else "")
_set_resume_from("scripts/train_ldm_local.sh", ldm_ckpt if have_ldm else "")
print(f"  [续训] train_vqvae_local.sh RESUME_FROM -> {vqvae_ckpt if have_vqvae else '(空，从零训练)'}")
print(f"  [续训] train_ldm_local.sh    RESUME_FROM -> {ldm_ckpt if have_ldm else '(空，从零训练)'}")

# 数据准备脚本按 CPU 核数调渲染并行度
cpu_cores = info["cpu_cores"]
render_workers = max(2, min(16, cpu_cores))
def set_sh_var(sh_path, var, value):
    with open(sh_path, encoding="utf-8") as f:
        lines = f.readlines()
    for i, ln in enumerate(lines):
        if ln.startswith(var + "="):
            body = ln[len(var)+1:].rstrip("\n")
            tail = ""
            if "#" in body:
                tail = "  " + body[body.index("#"):]
            lines[i] = f"{var}={value}{tail}\n"
            break
    with open(sh_path, "w", encoding="utf-8") as f:
        f.writelines(lines)

set_sh_var("scripts/prepare_dataset.sh", "NUM_WORKERS", render_workers)
# 本地机若无 CUDA，extract_charset 切 cpu（该步无实际张量计算）
set_sh_var("scripts/extract_charset.sh", "DEVICE", '"cuda"' if info["gpu_available"] else '"cpu"')

# 显存冗余量：写入本地训练脚本（None → auto，跟随 conservative 档）
from utils.hardware import auto_vqvae_batch_size, resolve_vram_reserve_fraction

vram_reserve_value = "auto" if VRAM_RESERVE_FRACTION is None else VRAM_RESERVE_FRACTION
set_sh_var("scripts/train_vqvae_local.sh", "VRAM_RESERVE_FRACTION", vram_reserve_value)
vram_frac = resolve_vram_reserve_fraction("conservative", VRAM_RESERVE_FRACTION)
est_batch = auto_vqvae_batch_size(info["vram_gb"], preset="conservative",
                                  vram_reserve_fraction=VRAM_RESERVE_FRACTION)
print(f"  显存预留比例: {vram_frac:.2f}（{'auto 跟随档位' if VRAM_RESERVE_FRACTION is None else 'Cell 0 指定'}）")
print(f"  显存 {info['vram_gb']:.1f} GB → 预计 VQ-VAE batch_size ≈ {est_batch}")

# 训练脚本（本地 conservative 档）
VQVAE_TRAIN_SCRIPT = "scripts/train_vqvae_local.sh"
LDM_TRAIN_SCRIPT = "scripts/train_ldm_local.sh"

print(f"\n[训练脚本] VQ-VAE → {VQVAE_TRAIN_SCRIPT}（conservative 本地档）")
print(f"[训练脚本] LDM     → {LDM_TRAIN_SCRIPT}（conservative 本地档）")
print(f"\n全部初始化完成！字体: {font_path}")

---
## Cell 2: 数据准备（纯 CPU，本地可直接运行）

> 分析字体覆盖率 → 渲染字形图片 → 提取训练/验证字符集。幂等，已完成会自动跳过。

In [ ]:
import os, json, subprocess

if not DO_DATA_PREP:
    print("DO_DATA_PREP=False，跳过 Cell 2")
else:
    def _load_state() -> dict:
        if os.path.exists(STATE_FILE):
            try:
                with open(STATE_FILE, "r", encoding="utf-8") as f:
                    return json.load(f)
            except Exception:
                return {}
        return {}

    def _save_state(s: dict) -> None:
        with open(STATE_FILE, "w", encoding="utf-8") as f:
            json.dump(s, f, ensure_ascii=False, indent=2)

    state = _load_state()
    data_done = os.path.isdir("data/reference") and os.path.isdir("data/target")
    splits_done = os.path.exists(f"charsets/splits/{FONT_NAME}/train.txt") and \
                  os.path.exists(f"charsets/splits/{FONT_NAME}/val.txt")
    same_font = state.get("data_font") == FONT_NAME

    if state.get("data_prep_done") and same_font and data_done and splits_done:
        print(f"字体 {FONT_NAME} 的数据准备已完成，跳过 Cell 2")
    else:
        print("\n===== 1. 分析字体覆盖率 =====")
        subprocess.run(["bash", "scripts/analyze_font.sh"], check=True)
        print("\n===== 2. 生成数据集图片 =====")
        subprocess.run(["bash", "scripts/prepare_dataset.sh"], check=True)
        print("\n===== 3. 提取训练/验证字符集 =====")
        subprocess.run(["bash", "scripts/extract_charset.sh"], check=True)

        def _png_count(d):
            return len([f for f in os.listdir(d) if f.endswith(".png")]) if os.path.isdir(d) else 0

        n_tgt, n_ref = _png_count("data/target"), _png_count("data/reference")
        print(f"\n===== 数据集自检 =====")
        print(f"  data/target:    {n_tgt} 张")
        print(f"  data/reference: {n_ref} 张")

        if n_tgt == 0:
            raise RuntimeError("data/target 为空：请检查 prepare_dataset.sh 的 TARGET_FONT_PATH 与字体文件")
        if n_ref == 0:
            raise RuntimeError(
                "data/reference 为空：fonts/jigmo/ 下没有可用的参考字体"
                "（jigmo.ttf / jigmo2.ttf / jigmo3.ttf）。\n"
                "        请重跑 Cell 1（会自动准备 Jigmo），或手动下载 "
                "https://kamichikoichi.github.io/jigmo/ 解压到 fonts/jigmo/ 后重跑 Cell 2。"
            )
        if not os.path.exists(f"charsets/splits/{FONT_NAME}/train.txt"):
            raise RuntimeError("train.txt 未生成，请检查 extract_charset.sh")

        state["data_prep_done"] = True
        state["data_font"] = FONT_NAME
        _save_state(state)
        print("\n===== 数据准备完成 =====")

---
## Cell 3: 训练 VQ-VAE（conservative 本地档，约 6-8 小时）

> 需要 NVIDIA GPU 或 Apple Silicon。断连/重启后重跑 Cell 0、Cell 1 即可从断点 epoch 精确续训。

In [ ]:
import os, subprocess

if not DO_TRAIN_VQVAE:
    print("DO_TRAIN_VQVAE=False，跳过 Cell 3")
elif not os.path.isdir("data"):
    print("data/ 不存在，请先运行 Cell 2 完成数据准备")
elif not check_training_viability()["viable"]:
    print("当前机器不满足训练条件（无 NVIDIA GPU / Apple Silicon），无法训练 VQ-VAE")
else:
    print(f"将执行训练脚本: {VQVAE_TRAIN_SCRIPT}（conservative 档，batch/workers 运行时自适应）")
    subprocess.run(["bash", VQVAE_TRAIN_SCRIPT], check=True)

---
## Cell 4-前置操作：离线放置 VGG16 权重（可选，联网下载慢时运行）

> **为什么需要**：Cell 4（训练 LDM 的 LPIPS 验证）与 Cell 5（LPIPS 指标）都会用到 VGG16 预训练权重（约 553MB），联网下载慢时可先手动下载放好。
>
> **把 `vgg16-397923af.pth` 直接丢进项目根目录即可**，本 Cell 会先直查这些固定位置（项目根 / `fonts/` / 上级目录 / 家目录 / 下载 / 桌面）——只做一次文件名拼接判断，**不遍历 C 盘 D 盘**；都没命中时才会再看一层子目录（可用 `SHALLOW_SEARCH=False` 关掉）。也可在 Cell 顶部直接填 `VGG16_SRC = "你的绝对路径"`，完全跳过搜索。
>
> 找到后移动到**本机** PyTorch Hub 缓存目录 `torch.hub.get_dir()/checkpoints`（Windows 通常为 `C:/Users/你/.cache/torch/hub/checkpoints`，macOS/Linux 为 `~/.cache/torch/hub/checkpoints`），后续直接命中、免下载。
>
> ⚠️ 这是 **VGG 预训练权重**，不是 `checkpoints/vqvae_{FONT_NAME}.pth`（本项目自己的模型），两者**不要混淆、不要放进 `checkpoints/`**。
>
> 📥 下载备份：`https://github.com/ICW-k/HanziGen_ICWfork/raw/main/vgg16-397923af.pth`
>
> 已联网且不在乎下载时间可跳过本 Cell，Cell 4 / Cell 5 会在首次使用时自动在线拉取。

In [ ]:
import os, shutil, torch

# ===== 离线放置 VGG16 权重：优先直查项目根目录，不遍历 C 盘 / D 盘 =====
VGG16_FILE = "vgg16-397923af.pth"

# ① 直接指定绝对路径（最省事，零搜索）：文件放哪儿就写哪儿
#    例：VGG16_SRC = "D:/weights/vgg16-397923af.pth"
VGG16_SRC = ""

# ② 直查目录（零遍历：只在这些目录里拼一次文件名判断存在性，瞬间完成）
DIRECT_DIRS = [
    os.getcwd(),                        # 项目根目录（最常见：直接丢在 HanziGen 根下）
    os.path.join(os.getcwd(), "fonts"), # 项目根/fonts
    os.path.dirname(os.getcwd()),       # 项目上级目录
    os.path.expanduser("~"),            # 家目录
    os.path.join(os.path.expanduser("~"), "Downloads"),   # 下载目录
    os.path.join(os.path.expanduser("~"), "Desktop"),     # 桌面
]

# ③ 上面都没命中时，是否允许再往下看一层子目录（仍不递归全盘）。不需要就设 False
SHALLOW_SEARCH = True

DST_DIR = os.path.join(torch.hub.get_dir(), "checkpoints")   # 跨平台：Windows / macOS / Linux 各自的用户缓存目录
os.makedirs(DST_DIR, exist_ok=True)
DST = os.path.join(DST_DIR, VGG16_FILE)

def locate(roots, depth: int):
    """depth=0: 只查 roots 本身；depth=1: 再看一层子目录。返回第一个命中的路径。"""
    for root in roots:
        if not os.path.isdir(root):
            continue
        if depth == 0:
            p = os.path.join(root, VGG16_FILE)
            if os.path.isfile(p):
                return p
            continue
        try:
            for entry in os.listdir(root):
                sub = os.path.join(root, entry)
                if not os.path.isdir(sub):
                    continue
                p = os.path.join(sub, VGG16_FILE)
                if os.path.isfile(p):
                    return p
        except (PermissionError, OSError):
            continue
    return None

if os.path.exists(DST):
    print(f"VGG16 权重已就位，跳过: {DST}")
else:
    src = VGG16_SRC if (VGG16_SRC and os.path.isfile(VGG16_SRC)) else None
    if src:
        print(f"[指定路径] 命中: {src}")
    else:
        src = locate(DIRECT_DIRS, depth=0)
        if src:
            print(f"[直查根目录] 命中: {src}")
        elif SHALLOW_SEARCH:
            src = locate(DIRECT_DIRS, depth=1)
            if src:
                print(f"[浅搜一层] 命中: {src}")

    if src is None:
        print(f"[WARN] 未找到 {VGG16_FILE}")
        print(f"  最省事：把文件放进项目根目录 {os.getcwd()}/ 后重跑本 Cell")
        print(f"  或在本 Cell 顶部填 VGG16_SRC = \"你的绝对路径/{VGG16_FILE}\"")
        print(f"  目标缓存位置: {DST}")
        print(f"  下载地址: https://github.com/ICW-k/HanziGen_ICWfork/raw/main/{VGG16_FILE}")
        print("  也可跳过本 Cell：Cell 4 / Cell 5 会在首次使用时自动联网下载（较慢）")
    else:
        if os.path.abspath(src) != os.path.abspath(DST):
            shutil.move(src, DST)
        print(f"VGG16 权重已就位: {DST}")
        print("  Cell 4 / Cell 5 的 LPIPS 将直接命中本地缓存，跳过远程下载")

---
## Cell 4: 训练 LDM（conservative 本地档，约 10-15 小时）

> 依赖 Cell 3 产物 `checkpoints/vqvae_{FONT_NAME}.pth`。

In [ ]:
import os, subprocess

vqvae_ckpt = f"checkpoints/vqvae_{FONT_NAME}.pth"

if not DO_TRAIN_LDM:
    print("DO_TRAIN_LDM=False，跳过 Cell 4")
elif not os.path.exists(vqvae_ckpt):
    print(f"{vqvae_ckpt} 不存在，请先完成 Cell 3 训练 VQ-VAE")
elif not check_training_viability()["viable"]:
    print("当前机器不满足训练条件（无 NVIDIA GPU / Apple Silicon），无法训练 LDM")
else:
    print("VQ-VAE 检查点确认：")
    print(f"  {vqvae_ckpt}")
    print(f"将执行训练脚本: {LDM_TRAIN_SCRIPT}（conservative 档，batch/workers 运行时自适应）")
    subprocess.run(["bash", LDM_TRAIN_SCRIPT], check=True)

---
## Cell 5: 推理生成 + 评估指标（GPU）

> 补字基准在 Cell 0 的 `CHARSET_BASE` 选择：`jf7000`（默认）/ `unihan` / `gbk`（简体 20,902 字）/ `gb2312`（核心 6,763 字）。
>
> 依赖 Cell 4 产物 `checkpoints/ldm_{FONT_NAME}.pth`。本 Cell 会自动把 `scripts/inference.sh` 的 `CHARSET_PATH` 指向所选基准的缺字表，并按本机硬件设置 `DEVICE`（cuda / mps / cpu）。
>
> SVG 转换已拆分到 **Cell 6**（纯 CPU）——本 Cell 结束后可切无卡模式再运行 Cell 6，省 GPU 机时。

In [ ]:
import os, re, subprocess
from utils.hardware import detect_hardware

ldm_ckpt = f"checkpoints/ldm_{FONT_NAME}.pth"

if not DO_INFERENCE:
    print("DO_INFERENCE=False，跳过 Cell 5")
elif not os.path.exists(ldm_ckpt):
    print(f"{ldm_ckpt} 不存在，请先完成 Cell 4 训练 LDM")
else:
    # ===== 1. 按补字基准确定字符集文件 =====
    def build_charset_path(base: str) -> str:
        if base in ("gbk", "gb2312"):
            # 用系统编码器直接计算 GBK / GB2312 缺失字，无需额外字集文件
            from fontTools.ttLib import TTFont
            base_chars = set()
            for cp in list(range(0x3400, 0x4DC0)) + list(range(0x4E00, 0xA000)):
                try:
                    chr(cp).encode(base)
                    base_chars.add(chr(cp))
                except UnicodeEncodeError:
                    pass
            font = TTFont(f"fonts/{TARGET_FONT}", fontNumber=0)
            cmap = set()
            for table in font["cmap"].tables:
                if table.isUnicode():
                    cmap.update(table.cmap.keys())
            missing = sorted(base_chars - {chr(c) for c in cmap})
            out_dir = f"charsets/{base}_coverage/{FONT_NAME}"
            os.makedirs(out_dir, exist_ok=True)
            out_path = f"{out_dir}/missing.txt"
            with open(out_path, "w", encoding="utf-8") as f:
                f.write("\n".join(missing))
            print(f"[{base}] 基准 {len(base_chars)} 字，字体缺失 {len(missing)} 字 → {out_path}")
            return out_path
        if base == "unihan":
            p = f"charsets/unihan_coverage/{FONT_NAME}/missing.txt"
        else:  # jf7000
            p = f"charsets/jf7000_coverage/{FONT_NAME}/missing.txt"
        if not os.path.exists(p):
            raise FileNotFoundError(f"{p} 不存在，请先运行 Cell 2 完成字体分析")
        return p

    charset_path = build_charset_path(CHARSET_BASE)
    print(f"补字基准: {CHARSET_BASE} → {charset_path}")

    # ===== 2. inference.sh 指向该字符集，并按本机硬件设置设备 =====
    info = detect_hardware()
    device = "cuda" if info["gpu_available"] else ("mps" if info["mps_available"] else "cpu")
    with open("scripts/inference.sh", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r'CHARSET_PATH="[^"]*"', f'CHARSET_PATH="{charset_path}"', content)
    content = re.sub(r'DEVICE="[^"]*"', f'DEVICE="{device}"', content)
    with open("scripts/inference.sh", "w", encoding="utf-8") as f:
        f.write(content)
    print(f"inference.sh 已指向补字基准字符集（DEVICE={device}）")

    # ===== 3. 推理生成（关键步骤） =====
    print("\n===== 推理生成 =====")
    subprocess.run(["bash", "scripts/inference.sh"], check=True)

    # ===== 补字数量自检 =====
    gen_dir = f"samples_{FONT_NAME}/inference/gen"
    if os.path.isdir(gen_dir):
        gen_pngs = [p for p in os.listdir(gen_dir) if p.endswith(".png")]
        print(f"\n===== 推理结果自检 =====")
        print(f"实际生成 PNG: {len(gen_pngs)} 张（位于 {gen_dir}/）")
    else:
        print(f"[WARN] 未找到生成目录 {gen_dir}")

    # 计算评估指标（非关键，失败不阻断）
    print("\n===== 计算评估指标 =====")
    with open("scripts/compute_metrics.sh", encoding="utf-8") as f:
        content = f.read()
    content = re.sub(r'DEVICE="[^"]*"', f'DEVICE="{device}"', content)
    with open("scripts/compute_metrics.sh", "w", encoding="utf-8") as f:
        f.write(content)
    try:
        subprocess.run(["bash", "scripts/compute_metrics.sh"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"[WARN] 评估指标计算失败（返回码 {e.returncode}），不影响补字结果。")

    print("\n===== GPU 阶段完成！SVG 转换请运行 Cell 6（纯 CPU，可切无卡模式） =====")

---
## Cell 6: SVG 转换（纯 CPU，可在无卡模式运行）

> 依赖 Cell 5 产物 `samples_{FONT_NAME}/inference/gen/`。与 GPU 无关——建议 GPU 实例跑完 Cell 5 后**切无卡模式**再运行本 Cell，省机时。
>
> 转换完成后可用 gbk notebook 的 Cell 4（或 cloudstudio 版 Cell 7/8）导出 zip 下载。

In [ ]:
import os, re, subprocess

# ===== CPU 核数检测与转换并行度分配 =====
try:
    cpu_cores = len(os.sched_getaffinity(0))   # Linux：本进程实际可用的核数
except AttributeError:
    cpu_cores = os.cpu_count() or 4            # Windows / macOS 回退
convert_workers = max(1, min(32, cpu_cores - 2))
print(f"检测到可用 CPU 核数: {cpu_cores} → SVG 转换进程数: {convert_workers}")

with open("scripts/convert_to_svg.sh", encoding="utf-8") as f:
    content = f.read()
content = re.sub(r"NUM_WORKERS=auto", f"NUM_WORKERS={convert_workers}", content)
with open("scripts/convert_to_svg.sh", "w", encoding="utf-8") as f:
    f.write(content)

png_dir = f"samples_{FONT_NAME}/inference/gen"
svg_dir = f"svgs_{FONT_NAME}"

if not os.path.isdir(png_dir):
    print(f"{png_dir} 不存在，请先在 GPU 实例完成 Cell 5 推理")
else:
    n_png = len([f for f in os.listdir(png_dir) if f.endswith(".png")])
    print(f"待转换 PNG: {n_png} 张（{png_dir}/）")
    print("\n===== SVG 转换（多进程并行，CPU 跑满后上万张约数分钟~数十分钟） =====")
    subprocess.run(["bash", "scripts/convert_to_svg.sh"], check=True)

    n_svg = len([f for f in os.listdir(svg_dir) if f.endswith(".svg")]) if os.path.isdir(svg_dir) else 0
    print("\n===== 转换完成 =====")
    print(f"SVG 输出: {n_svg} 个 → {svg_dir}/")
    print("下一步：gbk notebook 的 Cell 4（或 cloudstudio 版 Cell 7/8）导出 zip")

---
## 常见问题

| 问题 | 处理 |
|---|---|
| `FileNotFoundError: 'fonts/myfont.ttf'` | Cell 0 的 `TARGET_FONT` 还是占位默认值，或字体没放进 `fonts/`。改对后**重跑 Cell 0 + Cell 1**（Cell 1 会把字体路径写进所有脚本），再跑 Cell 2 |
| `ModuleNotFoundError: No module named 'xxx'` | 依赖没装到 kernel 所在环境。重跑 Cell 1（会自动补装）；或在 notebook 里执行 `import sys, subprocess; subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'])` |
| 装了包仍报找不到模块 | 终端 `python` 与 notebook kernel 不是同一个解释器（AutoDL / conda 常见）。Cell 1 会检测并打印两者路径，按提示激活同一环境后重启 kernel |
| Cell 1 提示「无法训练」 | 你的机器无 NVIDIA GPU / Apple Silicon（如仅 AMD 核显）。训练需改用云端 GPU 实例，数据准备与 SVG 转换仍可本地跑 |
| 训练 OOM | 优先在 Cell 0 调小 `VRAM_RESERVE_FRACTION`（如 0.70）后重跑 Cell 1 + Cell 3；也可手动调低 `scripts/train_vqvae_local.sh` 的 `BATCH_SIZE`（把 `auto` 改为具体整数） |
| 想少留冗余提速 | Cell 0 设 `VRAM_RESERVE_FRACTION = 0.92`，重跑 Cell 1 + Cell 3（仅影响 VQ-VAE batch，LDM 不受影响） |
| 想压榨性能 | 把 local 脚本的 `PRESET=conservative` 改为 `PRESET=aggressive`（会占用更多 CPU，适合专用训练机） |
| 只想补 GBK / GB2312 简体缺字 | Cell 0 设 `CHARSET_BASE="gbk"` 或 `"gb2312"`（训练完成后只跑 Cell 5 即可，会自动生成缺字表并改写 `inference.sh`） |
| 换基准后补字数量没变 | Cell 5 每次运行都会按 `CHARSET_BASE` 重新定位字表并改写 `scripts/inference.sh` 的 `CHARSET_PATH`，重跑 Cell 5 即可 |
| unihan / gbk 补字太慢 | `scripts/inference.sh` 的 `SAMPLE_STEPS=50` 改为 `20`（速度约 2.5 倍，质量略降） |
| 换字体后想重训 | 删除 `checkpoints/vqvae_{FONT_NAME}.pth`、`ldm_{FONT_NAME}.pth` 与 `colab_state.json`，重跑 Cell 1 |